In [1]:
import pathlib

import reskit as rk
import numpy as np
import pandas as pd


In [ ]:
# Create Placements DataFrame with turbine locations and specifications

geokit_df = pd.read_csv(pathlib.Path.cwd().joinpath("offshore_wind_turbine_depths.csv"))

geokit_df.rename(columns={"Latitude": "lat", "Longitude": "lon"}, inplace=True)
number_of_placements=geokit_df.shape[0]

depth_series = depth_series = geokit_df.loc[:, "depth"]

hub_height_series = pd.Series(data=[120] * number_of_placements,name="hub_height")
capacity_series = pd.Series(data=[4000] * number_of_placements, name="capacity")
rotor_diameter_series = pd.Series(data=[130] * number_of_placements, name="rotor_diam")
power_curve_series = pd.Series(data=["V117-3300"] * number_of_placements, name="powerCurve")

total_height_series = depth_series + hub_height_series

placements_df = pd.concat(
    [
        geokit_df.loc[:, "lat"],
        geokit_df.loc[:, "lon"],
        hub_height_series,
        capacity_series,
        rotor_diameter_series,
        power_curve_series,
    ],axis=1
)
print(placements_df.head(5))
placements = pd.DataFrame(
    {
        "lon": [5.985195, 5.994685, 6.004750],
        "lat": [50.797254, 50.794208, 50.784432],
        "hub_height": [120, 120, 82],
        "capacity": [4000, 4000, 4000],
        "rotor_diam": [130, 150, 136],
        "powerCurve": [np.nan, np.nan, "V117-3300"],
    }
)
# placements



          lat       lon  hub_height  capacity  rotor_diam powerCurve
0   51.590921  2.890734         120      4000         130  V117-3300
1   51.595143  2.899807         120      4000         130  V117-3300
2   51.586339  2.897291         120      4000         130  V117-3300
3   51.581578  2.904927         120      4000         130  V117-3300
4   51.568373  2.929721         120      4000         130  V117-3300
5   51.564061  2.938434         120      4000         130  V117-3300
6   51.594065  2.941129         120      4000         130  V117-3300
7   51.577805  2.931158         120      4000         130  V117-3300
8   51.573583  2.939063         120      4000         130  V117-3300
9   51.569451  2.946878         120      4000         130  V117-3300
10  51.582387  2.923522         120      4000         130  V117-3300
11  51.590831  2.927295         120      4000         130  V117-3300
12  51.586519  2.916156         120      4000         130  V117-3300
13  51.578883  2.948226         12

In [13]:

weather_year = 2018

workflow_args = {
    "placements": placements_df,
    "era5_path": f"/benchtop/shared_data/weather_data/processed_weather_data/ERA5_global_processed_V2022.02/4/<X-TILE>/<Y-TILE>/{weather_year}/reanalysis-era5-single-levels.z4.x<X-TILE>.y<Y-TILE>.y{weather_year}.*.nc",
    "gwa_100m_path": "/benchtop/internal/home/c-winkler/Research/01_Dissertation/03_RESkit/01_preprocessing/01_expand_GWA_to_EEZ/01_avg_annual_windspeed_100m_GWA_ERA5_interpolated.tif",
    "max_batch_size": 15000,
    "height_scaling_data": {
        10: "/fast/central/shared_data/Global_Wind_Atlas/GWA_4.0/wind_speed_cog_10m.tif",
        50: "/fast/central/shared_data/Global_Wind_Atlas/GWA_4.0/wind_speed_cog_50m.tif",
        100: "/fast/central/shared_data/Global_Wind_Atlas/GWA_4.0/wind_speed_cog_100m.tif",
        150: "/fast/central/shared_data/Global_Wind_Atlas/GWA_4.0/wind_speed_cog_150m.tif",
        200: "/fast/central/shared_data/Global_Wind_Atlas/GWA_4.0/wind_speed_cog_200m.tif",
    },
}


out = rk.execute_workflow_iteratively(
    workflow=rk.wind.wind_era5_PenaSanchezDunkelWinklerEtAl2025,
    weather_path_varname="era5_path",
    zoom=4,
    **workflow_args,
)

# extract the mean cfs for these 2 turbines
cfs = out.capacity_factor.values.mean(axis=0)


2026-01-23 19:59:12.339340 Now processing tile 1/4 with 13533 locations: /benchtop/shared_data/weather_data/processed_weather_data/ERA5_global_processed_V2022.02/4/8/5/2018/reanalysis-era5-single-levels.z4.x8.y5.y2018.*.nc
2026-01-23 20:03:24.431172 Now extracting correction factors for a total of 13533 placements from /fast/home/j-belina/RESKit/reskit/wind/core/data/cf_correction_factors_PSDW2025.tif:
2026-01-23 20:03:29.766212 Based on max_batch_size=13533, the total of 13533 placements were split into 1 sub batches. Proceeding with batch 1/1 (id=0) with 13533 placements.
2026-01-23 20:03:37.529277 Maximum rel. deviation after initial simulation is 0.2885, Number/share of placements with deviation > tolerance (0.01): 13533/13533. More iterations required.
2026-01-23 20:03:45.689478 Maximum rel. deviation after 1 additional iteration(s) is 0.1236, Number/share of placements with deviation > tolerance (0.01): 13533/13533. More iterations required.
2026-01-23 20:03:54.427963 Maximum rel

In [19]:
total_energy_per_turbine_mj=sum(out["capacity_factor"] * 60 * 60 * out["capacity"])
total_energy_per_turbine_mj
print("total_energy_per_turbine_mj: ", total_energy_per_turbine_mj)
total_energy_tj=sum(total_energy_per_turbine_mj) / 1e6
print("total_energy_tj: ",total_energy_tj)

total_energy_per_turbine_mj:  <xarray.DataArray (location: 19605)> Size: 157kB
array([5.66502250e+10, 5.66337139e+10, 5.66301154e+10, ...,
       5.84969495e+10, 5.84675119e+10, 5.84397677e+10])
Coordinates:
  * location  (location) int64 157kB 0 1 2 3 4 ... 19442 19443 19444 19445 19446
total_energy_tj:  <xarray.DataArray ()> Size: 8B
array(1.13618687e+09)
Coordinates:
    location  int64 8B 19446
